In [ ]:
dbutils.widgets.text("pipeline_name", "<pipeline_name>")
dbutils.widgets.text("connection_name", "<connection>")
dbutils.widgets.text("ingestion_dir", "<ingestion_dir>")
dbutils.widgets.text("dlt_ingest_limit", "-1")
dbutils.widgets.text("dev_mode", "false")
dbutils.widgets.text("dlt_env", "{}")

In [ ]:
INGESTION_DIR = dbutils.widgets.get("ingestion_dir")

In [ ]:
%pip install -q -c $INGESTION_DIR/../.github/managed-constraints.txt "dlt[databricks]" pyarrow vibedata-dlt-core vibedata-dlt-databricks-notebook -r $INGESTION_DIR/requirements.txt

In [ ]:
dbutils.library.restartPython()

In [ ]:
from vibedata.dlt.databricks import DatabricksSecretScopeProvider, DatabricksUcSecretsProvider, databricks_files_api, setup_environment

import dlt

setup_environment()
# Each resolves nothing unless config.toml names its store; the secret-scope provider keeps a scope-bound Domain's committed pipeline running.
dlt.config.register_provider(DatabricksUcSecretsProvider())
dlt.config.register_provider(DatabricksSecretScopeProvider())

In [ ]:
import json
import os

from vibedata.dlt.core import dataset, persist_audit

from contracts import COLUMNS  # generated by `odcs.py generate`; a resource with no entry raises
from sources.<connector> import <entry_point>

PIPELINE_NAME = dbutils.widgets.get("pipeline_name")
CONNECTION_NAME = dbutils.widgets.get("connection_name")
LIMIT = int(dbutils.widgets.get("dlt_ingest_limit"))

os.environ.update(json.loads(dbutils.widgets.get("dlt_env")))

destination = databricks_files_api()

pipeline = dlt.pipeline(
    pipeline_name=PIPELINE_NAME,
    destination=destination,
    dataset_name=dataset("<source_schema>"),
    dev_mode=dbutils.widgets.get("dev_mode") == "true",
)

source = <entry_point>.clone(name=CONNECTION_NAME, section=CONNECTION_NAME)().with_resources(<resource>, ...)
source.add_limit(LIMIT, count_rows=True)

source.<resource>.apply_hints(
    write_disposition="<replace|merge|append>",
    columns=COLUMNS["<resource>"],
    schema_contract={"columns": "freeze", "tables": "evolve", "data_type": "freeze"},
)

try:
    load_info = pipeline.run(source)
except Exception as exc:
    persist_audit(pipeline, pipeline.last_trace, error_message=str(exc))
    raise
print(load_info)
print("audit:", persist_audit(pipeline, pipeline.last_trace))

unfinished = [package for package in load_info.load_packages if package.state != "loaded"]
if load_info.has_failed_jobs or unfinished:
    raise RuntimeError("dlt load failed")

try:
    with pipeline.sql_client() as client:
        loaded_catalog = client.execute_sql("select current_catalog()")[0][0]
except Exception as exc:
    print("destination-identity unavailable:", exc)
else:
    dbutils.notebook.exit("destination-identity: " + json.dumps({"type": load_info.destination_type, "database": loaded_catalog, "dataset": pipeline.dataset_name}))